# 4. Inheritance

Inheritance in Python looks similar to Java, but with one massive difference: Python supports **multiple inheritance**. Java limits you to single class inheritance + interfaces; Python has no such restriction.

This notebook covers: basic inheritance, `super()`, method overriding, multiple inheritance, MRO (Method Resolution Order), type checking, `__init_subclass__`, and preventing inheritance.

### 4.1 Basic Inheritance & `super()`

**☕ JAVA:**
```java
public class Dog extends Animal {
    public Dog(String name, String breed) {
        super(name);       // Call parent constructor
        this.breed = breed;
    }
}
```

**🐍 PYTHON:** Use `class Dog(Animal):` instead of `extends`. Same `super()` concept, but no need to pass `self` to `super()` (Python 3 magic).

In [ ]:
class Animal:
    def __init__(self, name: str):
        self.name = name

    def speak(self) -> str:
        return "Some sound"

    def __str__(self) -> str:
        return f"{self.__class__.__name__}({self.name})"

class Dog(Animal):
    def __init__(self, name: str, breed: str):
        super().__init__(name)   # Call parent constructor
        self.breed = breed

dog = Dog("Buddy", "Labrador")
print(f"Name: {dog.name}")       # Inherited attribute
print(f"Breed: {dog.breed}")     # Own attribute
print(f"Speaks: {dog.speak()}")  # Inherited method
print(f"str: {dog}")             # Inherited __str__

### 4.2 Method Overriding

**☕ JAVA:** `@Override` annotation is optional but recommended.

**🐍 PYTHON:** No `@Override` annotation exists. Just define a method with the same name — it automatically overrides. You can call `super().method()` to invoke the parent's version.

In [ ]:
class Animal:
    def __init__(self, name: str):
        self.name = name

    def speak(self) -> str:
        return "..."

    def describe(self) -> str:
        return f"{self.name} says '{self.speak()}'"

class Dog(Animal):
    def speak(self) -> str:
        return "Woof!"       # Override — no annotation needed

class Cat(Animal):
    def speak(self) -> str:
        return "Meow!"

class Parrot(Animal):
    def speak(self) -> str:
        # Call parent version + add to it
        return f"{super().speak()} Polly wants a cracker!"

# Polymorphism — same as Java
animals: list[Animal] = [Dog("Buddy"), Cat("Whiskers"), Parrot("Polly")]
for animal in animals:
    print(f"  {animal.describe()}")

### 4.3 Multiple Inheritance — Java Can't Do This!

**☕ JAVA:** Only single class inheritance. Interfaces provide partial workaround:
```java
public class FlyingFish extends Fish implements Flyable { ... }
```

**🐍 PYTHON:** Full multiple inheritance — a class can inherit from **multiple** parent classes:
```python
class FlyingFish(Fish, Flyer): ...
```

In [ ]:
class Swimmer:
    def swim(self) -> str:
        return "🏊 Swimming!"

class Flyer:
    def fly(self) -> str:
        return "🦅 Flying!"

class Runner:
    def run(self) -> str:
        return "🏃 Running!"

# Multiple inheritance — gets ALL abilities
class Duck(Swimmer, Flyer, Runner):
    def __init__(self, name: str):
        self.name = name

duck = Duck("Donald")
print(f"{duck.name} can:")
print(f"  {duck.swim()}")
print(f"  {duck.fly()}")
print(f"  {duck.run()}")

### 4.4 MRO — Method Resolution Order

**☕ JAVA:** No need — single inheritance means there's only one path to resolve.

**🐍 PYTHON:** With multiple inheritance, the same method might exist in multiple parents. Python uses **C3 Linearization** to determine a consistent order. View it with `ClassName.__mro__` or `ClassName.mro()`.

The **Diamond Problem** occurs when class D inherits from B and C, which both inherit from A:

In [ ]:
#       A
#      / \
#     B   C
#      \ /
#       D

class A:
    def greet(self) -> str:
        return "Hello from A"

class B(A):
    def greet(self) -> str:
        return "Hello from B"

class C(A):
    def greet(self) -> str:
        return "Hello from C"

class D(B, C):   # Who wins? B or C?
    pass

d = D()
print(f"D says: {d.greet()}")   # B wins — listed first

# View the full resolution order:
print(f"\nMRO: {[cls.__name__ for cls in D.__mro__]}")

In [ ]:
# Cooperative multiple inheritance with super()
class A:
    def greet(self) -> str:
        return "A"

class B(A):
    def greet(self) -> str:
        return f"B → {super().greet()}"

class C(A):
    def greet(self) -> str:
        return f"C → {super().greet()}"

class D(B, C):
    def greet(self) -> str:
        return f"D → {super().greet()}"

# super() follows MRO, not direct parent!
print(f"Chain: {D().greet()}")   # D → B → C → A
print(f"MRO:   {[cls.__name__ for cls in D.__mro__]}")

> ⚠️ **Key insight:** `super()` doesn't always call the **direct parent** — it calls the **next class in the MRO**. This is why `B.greet()` calls `C.greet()` (not `A.greet()`) when called from `D`.

### 4.5 `isinstance()` vs `type()` — A Common Gotcha

**☕ JAVA:** `obj instanceof Dog` checks the full hierarchy.

**🐍 PYTHON:** `isinstance(obj, Dog)` and `issubclass(Dog, Animal)` both accept tuples for checking multiple types.

> ⚠️ **Gotcha:** `type(obj) == Class` checks the **exact** type only — it does NOT consider inheritance. Always prefer `isinstance()`.

In [ ]:
class Animal:
    pass

class Dog(Animal):
    pass

class Cat(Animal):
    pass

dog = Dog()

# isinstance — checks object type (including parents) ✅
print(f"isinstance(dog, Dog):    {isinstance(dog, Dog)}")       # True
print(f"isinstance(dog, Animal): {isinstance(dog, Animal)}")   # True
print(f"isinstance(dog, Cat):    {isinstance(dog, Cat)}")      # False

# Check multiple types at once with tuple
print(f"isinstance(dog, (Dog, Cat)): {isinstance(dog, (Dog, Cat))}")  # True

# ⚠️ type() — checks EXACT type only (ignores inheritance!)
print(f"\ntype(dog) == Dog:    {type(dog) == Dog}")       # True
print(f"type(dog) == Animal: {type(dog) == Animal}")     # ❌ FALSE! Dog != Animal

# issubclass — checks class hierarchy
print(f"\nissubclass(Dog, Animal): {issubclass(Dog, Animal)}")   # True
print(f"issubclass(Animal, Dog): {issubclass(Animal, Dog)}")     # False

### 4.6 `__init_subclass__` — Hook Into Subclassing

**☕ JAVA:** No equivalent — you can't execute code when a class is subclassed.

**🐍 PYTHON:** `__init_subclass__` is called automatically when a class is subclassed. Great for plugin registration, validation, or automatic setup.

In [ ]:
# Auto-registration pattern — plugins register themselves!
class Plugin:
    """Base class that auto-registers all plugins."""
    _registry: dict[str, type] = {}

    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        Plugin._registry[cls.__name__] = cls
        print(f"  📦 Registered plugin: {cls.__name__}")

    @classmethod
    def get_plugins(cls) -> dict[str, type]:
        return dict(cls._registry)

# These register themselves automatically — no decorator or function call needed!
class CSVExporter(Plugin):
    def export(self, data): return "CSV..."

class JSONExporter(Plugin):
    def export(self, data): return "JSON..."

class XMLExporter(Plugin):
    def export(self, data): return "XML..."

print(f"\nAll plugins: {list(Plugin.get_plugins().keys())}")

In [ ]:
# Enforcing constraints on subclasses
class Serializable:
    """Base class that forces subclasses to define 'format_name'."""

    def __init_subclass__(cls, format_name: str = "", **kwargs):
        super().__init_subclass__(**kwargs)
        if not format_name:
            raise TypeError(f"{cls.__name__} must specify format_name")
        cls.format_name = format_name

class JsonDoc(Serializable, format_name="json"):
    pass

class XmlDoc(Serializable, format_name="xml"):
    pass

print(f"JsonDoc format: {JsonDoc.format_name}")
print(f"XmlDoc format:  {XmlDoc.format_name}")

try:
    class BadDoc(Serializable):   # No format_name!
        pass
except TypeError as e:
    print(f"\n❌ {e}")

### 4.7 Preventing Inheritance — `@final`

**☕ JAVA:** `final class String { ... }` — cannot be subclassed.

**🐍 PYTHON:** Python 3.8+ provides `typing.final` for two purposes:

| Use | Java | Python |
|-----|------|--------|
| Prevent subclassing | `final class Foo` | `@final class Foo` |
| Prevent overriding | `final void method()` | `@final def method()` |

> ⚠️ `@final` is a **type-checker hint only** (enforced by mypy/pyright). Python itself won't stop subclassing at runtime. For runtime enforcement, combine with `__init_subclass__`.

In [ ]:
from typing import final

class Base:
    @final
    def critical_method(self) -> str:
        """This method should NOT be overridden."""
        return "Don't touch me!"

    def extensible_method(self) -> str:
        """This one is fine to override."""
        return "Override me!"

class Child(Base):
    # This would trigger a mypy/pyright error:
    # error: Cannot override final method "critical_method"
    # def critical_method(self) -> str: ...   # ❌ type checker flags this

    def extensible_method(self) -> str:
        return "Overridden! ✅"

c = Child()
print(f"Critical:    {c.critical_method()}")
print(f"Extensible:  {c.extensible_method()}")

In [ ]:
# Runtime enforcement — combine @final with __init_subclass__
class FinalClass:
    """Cannot be subclassed — enforced at runtime."""

    def __init_subclass__(cls, **kwargs):
        raise TypeError(f"{cls.__name__} cannot subclass FinalClass — it is final!")

try:
    class Attempt(FinalClass):
        pass
except TypeError as e:
    print(f"❌ {e}")

---

## 🧪 Try It Yourself

**Exercise 1:** Create a `Vehicle` base class with `make` and `year`. Create `Car(Vehicle)` and `Truck(Vehicle)` subclasses that override a `describe()` method.

In [ ]:
# Exercise 1: Your code here


**Exercise 2:** Create a diamond inheritance: `class ElectricAmphibious(ElectricVehicle, AmphibiousVehicle)` where both parents inherit from `Vehicle`. Print the MRO.

In [ ]:
# Exercise 2: Your code here


**Exercise 3:** Create a `Handler` base class that auto-registers subclasses using `__init_subclass__`. Each subclass should specify a `handles` keyword (e.g., `class ImageHandler(Handler, handles='image')`). Add a class method `get_handler(media_type)` that returns the right handler.

In [ ]:
# Exercise 3: Your code here


---

## 📝 Key Takeaways: Java → Python

| Concept | Java | Python |
|---------|------|--------|
| Inherit | `class Dog extends Animal` | `class Dog(Animal):` |
| Call parent constructor | `super(name)` | `super().__init__(name)` |
| Override | `@Override` annotation | Just redefine the method |
| Call parent method | `super.method()` | `super().method()` |
| Multiple inheritance | ❌ Not supported | ✅ `class D(B, C):` |
| Interfaces | `implements Runnable` | Not needed — use ABC or Protocol |
| Diamond problem | Can't happen | Solved by MRO (C3 linearization) |
| `super()` target | Always direct parent | Next in MRO (may skip parent!) |
| Type check | `obj instanceof Dog` | `isinstance(obj, Dog)` |
| Exact type check | N/A | `type(obj) == Dog` (no inheritance!) |
| Multi-type check | Chained `instanceof` | `isinstance(obj, (Dog, Cat))` |
| Class check | `Dog.class.isAssignableFrom(...)` | `issubclass(Dog, Animal)` |
| Hook into subclassing | Not possible | `__init_subclass__(**kwargs)` |
| Prevent subclassing | `final class Foo` | `@final` (type-checker) or `__init_subclass__` |
| Prevent overriding | `final void method()` | `@final` (type-checker only) |